# Notebook 02 — Comprehension Phase: Backbone Fine-Tuning & Selection

**Pipeline position:** Phase 1 (Comprehension) of the ArchPipeline two-phase transfer-learning framework — fine-tunes and compares 6 candidate backbones on the architecture-comprehension task (ACME/AADL), then selects the best backbone(s) to carry forward into Phase 2 (Evolution, notebook `04_evolution_finetuning.ipynb`).

**Inputs:** `train_comprehsion.jsonl`, `val_comprehsion.jsonl`, `test_comprehension_v2.jsonl` produced by `01_comprehension_dataset.ipynb` (paths centralized via `BASE_DIR`, Step 0C below).

**Models compared:** T5-base, CodeT5-base, UniXcoder, FLAN-T5-base, CodeT5+ 770M, BART-large.

**Outputs:** one fine-tuned checkpoint per model under `{BASE_DIR}/finetuned_*`, per-model metric JSON files and a global recap under `{BASE_DIR}/evaluation_comprehension/`, and a backbone-selection report (`sensitivity_backbone_selection.json`) used to choose the backbone(s) for Phase 2.

**Hardware:** single NVIDIA Tesla T4 GPU (Google Colab).

> **Note on this cleaned version.** Code comments, docstrings, print messages and identifiers have been translated from French to English for public release. Models were **not re-executed** for this cleanup (training is long-running and GPU-bound), so the recorded cell *outputs* below are the original Colab logs and may still contain French text — this is expected and does not affect the saved checkpoints, metrics, or results, which are unchanged from the original run.


In [ ]:
# ============================================================
# STEP 0A — Install dependencies
# ============================================================
!pip install rouge_score bert_score nltk transformers              datasets accelerate sentencepiece -q

from google.colab import drive
import os, json, torch, numpy as np

drive.mount('/content/drive')
print(f"✅ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


In [ ]:
# ============================================================
# STEP 0B — NLTK downloads (full set)
# ============================================================
import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
print("✅ NLTK ready")


In [ ]:
# ============================================================
# STEP 0C — Centralized paths (edit BASE_DIR once if needed)
# ============================================================
BASE_DIR  = "/content/drive/MyDrive"
CLEAN_DIR = f"{BASE_DIR}/dataset_clean"

# NOTE: "comprehsion" (sic) reproduces the actual filenames used during the
# original experiments and stored on Drive/Zenodo. Kept as-is for reproducibility.
TRAIN_PATH = f"{CLEAN_DIR}/train_comprehsion.jsonl"
VAL_PATH   = f"{CLEAN_DIR}/val_comprehsion.jsonl"
TEST_PATH  = f"{CLEAN_DIR}/test_comprehension_v2.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity check
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    status = "✅" if os.path.exists(p) else "❌"
    print(f"  {status} {p}")


## Step 1 — Shared utilities


In [ ]:
# ============================================================
# STEP 1A — JSONL loading
# ============================================================
def load_jsonl(path: str) -> list:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def load_datasets():
    """Loads the 3 splits (train/val/test) and prints basic statistics."""
    train = load_jsonl(TRAIN_PATH)
    val   = load_jsonl(VAL_PATH)
    test  = load_jsonl(TEST_PATH)
    print(f"📊 Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
    return train, val, test


In [ ]:
# ============================================================
# STEP 1B — Tokenizer preprocessing
# NOTE: no padding is applied here; DataCollatorForSeq2Seq handles
# dynamic padding per batch instead.
# ============================================================
def build_preprocess_fn(tokenizer,
                         max_input : int = 256,
                         max_target: int = 128):
    """
    Returns a preprocessing function adapted to the given tokenizer.

    Padding is intentionally omitted here: it is delegated to
    DataCollatorForSeq2Seq for dynamic per-batch padding, which
    reduces memory usage and avoids biasing the loss with
    unnecessary padding tokens.
    """
    def preprocess(examples):
        # Tokenize inputs — no padding
        model_inputs = tokenizer(
            examples["input"],
            max_length=max_input,
            truncation=True
        )
        # Tokenize targets — no padding
        labels = tokenizer(
            examples["target"],
            max_length=max_target,
            truncation=True
        )
        # Replace pad_token_id with -100 so these positions
        # are ignored by the cross-entropy loss
        model_inputs["labels"] = [
            [(t if t != tokenizer.pad_token_id else -100)
             for t in label]
            for label in labels["input_ids"]
        ]
        return model_inputs

    return preprocess


def tokenize_datasets(train_ds, val_ds, test_ds, tokenizer):
    """
    Applies preprocessing to the 3 splits.
    NOTE: removed columns are filtered dynamically to avoid a
    KeyError if a given column is absent from the dataset.
    """
    CANDIDATE_COLUMNS = [
        "input", "target", "adl_type",
        "element_type", "quality", "source"
    ]
    columns = [c for c in CANDIDATE_COLUMNS
               if c in train_ds.column_names]

    preprocess = build_preprocess_fn(tokenizer)

    tr = train_ds.map(preprocess, batched=True,
                      remove_columns=columns)
    v  = val_ds.map(preprocess, batched=True,
                    remove_columns=columns)
    te = test_ds.map(preprocess, batched=True,
                     remove_columns=columns)

    print(f"✅ Removed columns: {columns}")
    return tr, v, te


In [ ]:
# ============================================================
# STEP 1C — Training arguments
# Two profiles: standard (small models) and large (bigger models)
# ============================================================
from transformers import TrainingArguments

def get_training_args(output_dir : str,
                      large_model  : bool = False,
                      num_epochs   : int  = 20) -> TrainingArguments:
    """
    Returns TrainingArguments adapted to the model size.

    large_model=True  -> FLAN-T5-Large, CodeT5+ 770M
        batch_size=4 + gradient_accumulation=2
        (effective batch size = 8, no OOM on a 15GB T4)

    large_model=False -> T5-base, CodeT5-base
        batch_size=8 directly
    """
    if large_model:
        batch_train = 4
        batch_eval  = 4
        grad_accum  = 2
    else:
        batch_train = 8
        batch_eval  = 8
        grad_accum  = 1

    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = num_epochs,

        per_device_train_batch_size = batch_train,
        per_device_eval_batch_size  = batch_eval,
        gradient_accumulation_steps = grad_accum,

        learning_rate               = 1e-4,
        warmup_steps                = 100,
        weight_decay                = 0.01,

        eval_strategy                = "epoch",
        logging_strategy             = "epoch",
        save_strategy                = "epoch",
        save_total_limit             = 3,

        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,

        report_to                   = "none",
        fp16                        = torch.cuda.is_available(),
        seed                        = 42,
    )


In [ ]:
# ============================================================
# STEP 1D — Generic training pipeline
# ============================================================
from transformers import (
    Trainer, DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from datasets import Dataset

def run_training(model, tokenizer,
                  train_tok, val_tok,
                  training_args) -> Trainer:
    """
    Launches training with:
    - DataCollatorForSeq2Seq (dynamic per-batch padding)
    - EarlyStoppingCallback (patience=3, threshold=0.001)
    """
    collator = DataCollatorForSeq2Seq(
        tokenizer, model=model,
        padding=True,          # dynamic padding here
        label_pad_token_id=-100
    )

    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = train_tok,
        eval_dataset     = val_tok,
        data_collator    = collator,
        processing_class = tokenizer,
        callbacks        = [EarlyStoppingCallback(
            early_stopping_patience  = 3,
            early_stopping_threshold = 0.001
        )],
    )

    trainer.train()

    print(f"\n✅ Stopped at epoch: {trainer.state.epoch:.0f}/20")
    print(f"✅ Best eval_loss: {trainer.state.best_metric:.4f}")
    return trainer


In [ ]:
# ============================================================
# STEP 1E (legacy variant) — evaluation helper kept for reference.
# Superseded by evaluate_model() defined below; not called anywhere
# in this notebook, kept only to preserve the original development
# history of the evaluation function across notebooks 02/04.
# ============================================================
def evaluate_model_legacy(model, tokenizer, test_data, model_name="Model", output_dir="."):
    model.eval()
    predictions, references = [], []

    print(f"🔄 Generating predictions for {len(test_data)} examples...")
    for i, ex in enumerate(test_data):
        inputs = tokenizer(str(ex["input"]), return_tensors="pt", max_length=256, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        predictions.append(pred)
        references.append(str(ex["target"]).strip())

        if (i + 1) % 50 == 0:
            print(f"   {i+1}/{len(test_data)} generated...")

    print(f"✅ {len(predictions)} predictions generated")

    # Standard metrics
    em = sum(1 if p == r else 0 for p, r in zip(predictions, references)) / len(predictions)

    smooth = SmoothingFunction()
    bleu = corpus_bleu([[r.split()] for r in references], [p.split() for p in predictions], smoothing_function=smooth.method4)
    smooth_bleu = corpus_bleu([[r.split()] for r in references], [p.split() for p in predictions], smoothing_function=smooth.method1)

    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, r in zip(predictions, references):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    rouge1 = float(np.mean(r1))
    rouge2 = float(np.mean(r2))
    rougeL = float(np.mean(rL))

    meteor = float(np.mean([meteor_score([r.split()], p.split()) for p, r in zip(predictions, references)]))

    # Corrected BERTScore
    print("\n🔄 Computing BERTScore...")
    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(
            predictions, references,
            lang="en",
            model_type="roberta-base",      # more stable than "large"
            device=device,
            batch_size=16,
            verbose=False
        )
        bert_p  = float(P.mean().item())
        bert_r  = float(R.mean().item())
        bert_f1 = float(F1.mean().item())
    except Exception as e:
        print(f"⚠️ BERTScore failed: {e}")
        bert_p = bert_r = bert_f1 = 0.0

    # Display
    print(f"\n{'='*80}")
    print(f"📊 RESULTS — {model_name}")
    print(f"{'='*80}")
    for name, val in [
        ("Exact Match", em), ("BLEU", bleu), ("Smooth BLEU", smooth_bleu),
        ("ROUGE-1", rouge1), ("ROUGE-2", rouge2), ("ROUGE-L", rougeL),
        ("METEOR", meteor),
        ("BERT-Precision", bert_p), ("BERT-Recall", bert_r), ("BERT-F1", bert_f1),
    ]:
        print(f"  {name:<20} : {val:.4f}")
    print(f"{'='*80}")

    # Save
    results = {
        "model": model_name,
        "EM": em, "BLEU": bleu, "Smooth_BLEU": smooth_bleu,
        "ROUGE-1": rouge1, "ROUGE-2": rouge2, "ROUGE-L": rougeL,
        "METEOR": meteor,
        "BERT-P": bert_p, "BERT-R": bert_r, "BERT-F1": bert_f1,
    }
    path = f"{output_dir}/results_{model_name.replace(' ','_')}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    print(f"💾 Results saved: {path}")

    return results


In [ ]:
# ============================================================
# STEP 1E — Full evaluation (all metrics) — used throughout the notebook
# ============================================================
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from bert_score import score as bert_score_fn
from nltk.translate.meteor_score import meteor_score

def evaluate_model(model, tokenizer,
                    test_data : list,
                    model_name: str,
                    output_dir: str) -> dict:
    """
    Generates predictions on the test set and computes:
    EM, BLEU, ROUGE-1/2/L, METEOR, BERTScore (P/R/F1).
    """
    model.eval()
    predictions, references = [], []

    print(f"\n🔄 Generating predictions ({len(test_data)} examples)...")
    for i, example in enumerate(test_data):
        inputs = tokenizer(
            str(example["input"]),
            return_tensors="pt",
            max_length=256,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append(str(example["target"]))

        if (i + 1) % 20 == 0:
            print(f"   {i+1}/{len(test_data)} generated...")

    print(f"✅ {len(predictions)} predictions generated")

    # --- Exact Match ---
    em = sum(1 if p.strip() == r.strip() else 0
             for p, r in zip(predictions, references)) / len(predictions)

    # --- BLEU ---
    smooth      = SmoothingFunction()
    refs_bleu   = [[r.split()] for r in references]
    preds_bleu  = [p.split() for p in predictions]
    bleu        = corpus_bleu(refs_bleu, preds_bleu,
                              smoothing_function=smooth.method4)
    smooth_bleu = corpus_bleu(refs_bleu, preds_bleu,
                              smoothing_function=smooth.method1)

    # --- ROUGE ---
    scorer_r = rs_module.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, r in zip(predictions, references):
        s = scorer_r.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    rouge1 = float(np.mean(r1))
    rouge2 = float(np.mean(r2))
    rougeL = float(np.mean(rL))

    # --- METEOR ---
    meteor_scores = []
    for p, r in zip(predictions, references):
        try:
            meteor_scores.append(
                meteor_score([r.split()], p.split()))
        except Exception:
            meteor_scores.append(0.0)
    meteor = float(np.mean(meteor_scores))

    # --- BERTScore ---
    print("\n🔄 Computing BERTScore...")
    P, R, F1 = bert_score_fn(
        predictions, references, lang="en", verbose=False)
    bert_p  = float(P.mean().item())
    bert_r  = float(R.mean().item())
    bert_f1 = float(F1.mean().item())

    # --- Display ---
    print(f"\n{'='*60}")
    print(f"📊 RESULTS — {model_name}")
    print(f"{'='*60}")
    metrics = [
        ("Exact Match",   em),
        ("BLEU",          bleu),
        ("Smooth BLEU",   smooth_bleu),
        ("ROUGE-1",       rouge1),
        ("ROUGE-2",       rouge2),
        ("ROUGE-L",       rougeL),
        ("METEOR",        meteor),
        ("BERT-Precision",bert_p),
        ("BERT-Recall",   bert_r),
        ("BERT-F1",       bert_f1),
    ]
    for name, val in metrics:
        print(f"  {name:<20} : {val:.4f}")
    print(f"{'='*60}")

    # --- Save ---
    results = {
        "model"       : model_name,
        "EM"          : em,
        "BLEU"        : bleu,
        "Smooth_BLEU" : smooth_bleu,
        "ROUGE-1"     : rouge1,
        "ROUGE-2"     : rouge2,
        "ROUGE-L"     : rougeL,
        "METEOR"      : meteor,
        "BERT-P"      : bert_p,
        "BERT-R"      : bert_r,
        "BERT-F1"     : bert_f1,
    }
    path_json = f"{output_dir}/results_{model_name.replace(' ', '_')}.json"
    with open(path_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    print(f"\n💾 Results saved: {path_json}")
    return results


---
## 🔵 Model 1 — T5-base (220M)

**Reference model.** Standard T5 architecture, pretrained on C4.
Good starting point to validate the pipeline before moving to heavier models.


In [ ]:
# ============================================================
# MODEL 1 — T5-base
# ============================================================
from transformers import T5TokenizerFast, T5ForConditionalGeneration

OUTPUT_T5BASE = f"{BASE_DIR}/finetuned_T5base"
os.makedirs(OUTPUT_T5BASE, exist_ok=True)

# --- Load data ---
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# --- Model & tokenizer ---
print("⏳ Loading T5-base...")
tokenizer_t5 = T5TokenizerFast.from_pretrained("t5-base")
model_t5     = T5ForConditionalGeneration.from_pretrained(
    "t5-base").to(device)
print(f"✅ T5-base loaded | Params: "
      f"{sum(p.numel() for p in model_t5.parameters())/1e6:.0f}M")

# --- Tokenization ---
train_tok, val_tok, test_tok = tokenize_datasets(
    train_ds, val_ds,
    Dataset.from_list(test_data),
    tokenizer_t5
)

# --- Training ---
args_t5 = get_training_args(OUTPUT_T5BASE, large_model=False)
print("\n🚀 Starting T5-base fine-tuning...")
trainer_t5 = run_training(
    model_t5, tokenizer_t5,
    train_tok, val_tok, args_t5
)

trainer_t5.save_model(OUTPUT_T5BASE)
tokenizer_t5.save_pretrained(OUTPUT_T5BASE)
print(f"✅ Model saved: {OUTPUT_T5BASE}")


---
## 🟢 Model 2 — CodeT5-base (220M)

**Code-specialized model.** Pretrained on CodeSearchNet (6 programming languages).
Expected to better capture the formal syntactic structures of ACME/AADL.


In [ ]:
# ============================================================
# FIX — CodeT5-base tokenizer
# Resolves two errors:
#   1. ValueError: special_tokens_map contains objects instead
#      of plain strings
#   2. TypeError: extra_special_tokens incorrectly formatted
# Run ONCE before the CodeT5 training cell below.
# ============================================================
!pip install sentencepiece -q   # missing -> root cause of the main error

import json, os

# ── Locate the HuggingFace cache ────────────────────────────
CACHE_BASE = os.path.expanduser(
    "~/.cache/huggingface/hub/models--Salesforce--codet5-base"
)

def find_snapshots(cache_base: str) -> list:
    snap_dir = os.path.join(cache_base, "snapshots")
    if not os.path.exists(snap_dir):
        return []
    return [
        os.path.join(snap_dir, d)
        for d in os.listdir(snap_dir)
        if os.path.isdir(os.path.join(snap_dir, d))
    ]

snapshots = find_snapshots(CACHE_BASE)

if not snapshots:
    # Model hasn't been downloaded yet — force an initial download
    print("⏳ First CodeT5 download to populate the cache...")
    from transformers import AutoTokenizer
    try:
        AutoTokenizer.from_pretrained("Salesforce/codet5-base")
    except Exception:
        pass
    snapshots = find_snapshots(CACHE_BASE)

print(f"📁 {len(snapshots)} snapshot(s) found")

# ── Fix each snapshot ────────────────────────────────────────
def flatten_token(t) -> str:
    """Converts an AddedToken object or dict into a plain string."""
    if isinstance(t, dict):
        return t.get("content", str(t))
    return str(t)

FIXED_COUNT = 0

for snap in snapshots:
    stm_path = os.path.join(snap, "special_tokens_map.json")
    if not os.path.exists(stm_path):
        continue

    with open(stm_path) as f:
        content = json.load(f)

    fixed = {}
    for key, val in content.items():
        if key == "additional_special_tokens":
            # Convert each element into a plain string
            fixed[key] = [flatten_token(x) for x in val]
        elif isinstance(val, dict):
            # e.g. {"content": "<pad>", "single_word": false, ...}
            fixed[key] = flatten_token(val)
        else:
            fixed[key] = val

    with open(stm_path, "w") as f:
        json.dump(fixed, f, indent=2)

    FIXED_COUNT += 1
    print(f"✅ Fixed: {stm_path}")
    print(f"   Special tokens: {[k for k in fixed if k != 'additional_special_tokens']}")
    print(f"   extra_ids      : {len(fixed.get('additional_special_tokens', []))} tokens")

if FIXED_COUNT == 0:
    print("⚠️  No snapshot found after download.")
    print("   Re-run this cell after the CodeT5 cell (first download).")
else:
    print(f"\n✅ {FIXED_COUNT} file(s) fixed — re-run the CodeT5 training cell")

# ── Quick sanity check ────────────────────────────────────────
print("\n🔍 Testing tokenizer load...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "Salesforce/codet5-base",
    local_files_only=True
)
test = tokenizer("Component Type OSIComp = { Port p; }")
print(f"✅ Tokenizer OK — {len(test['input_ids'])} tokens generated")
print(f"   Type: {type(tokenizer).__name__}")


In [ ]:
# ============================================================
# MODEL 2 — CodeT5-base
# ============================================================
from transformers import AutoTokenizer, T5ForConditionalGeneration

OUTPUT_CODET5 = f"{BASE_DIR}/finetuned_CodeT5base"
os.makedirs(OUTPUT_CODET5, exist_ok=True)

# --- Load data (clean reload) ---
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# --- Model & tokenizer ---
print("⏳ Loading CodeT5-base...")
tokenizer_ct5 = AutoTokenizer.from_pretrained(
    "Salesforce/codet5-base",
    local_files_only=True   # ← force use of the fixed cache
)
model_ct5     = T5ForConditionalGeneration.from_pretrained(
    "Salesforce/codet5-base").to(device)
print(f"✅ CodeT5-base loaded | Params: "
      f"{sum(p.numel() for p in model_ct5.parameters())/1e6:.0f}M")

# --- Tokenization ---
train_tok, val_tok, _ = tokenize_datasets(
    train_ds, val_ds,
    Dataset.from_list(test_data),
    tokenizer_ct5
)

# --- Training ---
args_ct5 = get_training_args(OUTPUT_CODET5, large_model=False)
print("\n🚀 Starting CodeT5-base fine-tuning...")
trainer_ct5 = run_training(
    model_ct5, tokenizer_ct5,
    train_tok, val_tok, args_ct5
)

trainer_ct5.save_model(OUTPUT_CODET5)
tokenizer_ct5.save_pretrained(OUTPUT_CODET5)
print(f"✅ Model saved: {OUTPUT_CODET5}")


---
## 🟡 Model 3 — UniXcoder (~220M)

**Encoder-only architecture repurposed as Encoder-Decoder.** Pretrained for code understanding (GraphCodeBERT/UniXcoder family); rebuilt here as an `EncoderDecoderModel` since it has no native decoder.


In [ ]:
# ============================================================
# UniXcoder — Clean restart from the base architecture
# ============================================================
import os, gc, torch
from transformers import (
    RobertaTokenizer,
    EncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import Dataset

OUTPUT_UNI = f"{BASE_DIR}/finetuned_UniXcoder"

gc.collect()
print("🧹 RAM cleared")

train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

tokenizer = RobertaTokenizer.from_pretrained("microsoft/unixcoder-base")

print("⏳ Rebuilding the EncoderDecoder model...")
model_uni = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "microsoft/unixcoder-base",
    "microsoft/unixcoder-base"
)

# Important configuration
model_uni.config.decoder_start_token_id = tokenizer.cls_token_id
model_uni.config.pad_token_id           = tokenizer.pad_token_id
model_uni.config.eos_token_id           = tokenizer.sep_token_id

generation_config = model_uni.generation_config
generation_config.decoder_start_token_id = tokenizer.cls_token_id
generation_config.pad_token_id           = tokenizer.pad_token_id
generation_config.eos_token_id           = tokenizer.sep_token_id
generation_config.max_length           = 64
generation_config.num_beams            = 4
generation_config.early_stopping       = True
generation_config.no_repeat_ngram_size = 3
generation_config.length_penalty       = 2.0

model_uni.generation_config = generation_config

print("✅ Model rebuilt")

# Preprocessing
def preprocess_uni(examples):
    inputs = ["" + " " + str(x) for x in examples["input"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=False)
    labels = tokenizer([str(t) for t in examples["target"]], max_length=64, truncation=True, padding=False)
    model_inputs["labels"] = [
        [t if t != tokenizer.pad_token_id else -100 for t in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_tok = train_ds.map(preprocess_uni, batched=True, remove_columns=["input","target","adl_type","element_type","quality","source"])
val_tok   = val_ds.map(preprocess_uni,   batched=True, remove_columns=["input","target","adl_type","element_type","quality","source"])

print("✅ Tokenization done")

# Training arguments
args_uni = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_UNI,
    num_train_epochs            = 15,                    # reduced to 15 instead of 20
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,
    learning_rate               = 3e-5,
    warmup_steps                = 100,
    weight_decay                = 0.01,

    eval_strategy               = "epoch",
    logging_strategy            = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    predict_with_generate       = True,
    generation_max_length       = 64,
    generation_num_beams        = 4,

    fp16                        = False,
    report_to                   = "none",
    seed                        = 42,
    dataloader_num_workers      = 0,
)

trainer_uni = Seq2SeqTrainer(
    model            = model_uni,
    args             = args_uni,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    data_collator    = DataCollatorForSeq2Seq(tokenizer, model=model_uni, padding=True, label_pad_token_id=-100),
    processing_class = tokenizer,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
)

print("\n🚀 Restarting training from scratch...")
trainer_uni.train()

print(f"\n✅ Training finished at epoch: {trainer_uni.state.epoch:.0f}/15")


---
## 🟣 Model 4 — FLAN-T5-base (250M)

**Instruction-tuned T5.** Two cells follow: the initial training run, and a checkpoint-resume cell used after a Colab disconnect interrupted the first run partway through.


In [ ]:
# ============================================================
# FLAN-T5-base — Initial training run
# ============================================================
import os, gc, torch
from transformers import (
    AutoTokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_FLAN  = "google/flan-t5-base"
OUTPUT_FLAN = f"{BASE_DIR}/finetuned_flan_t5_base"
device      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_FLAN, exist_ok=True)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Data ─────────────────────────────────────────────────────
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# ── Tokenizer + fp32 model ───────────────────────────────────
print(f"Loading {MODEL_FLAN}...")
tokenizer_flan = AutoTokenizer.from_pretrained(MODEL_FLAN)
model_flan = T5ForConditionalGeneration.from_pretrained(
    MODEL_FLAN, torch_dtype=torch.float32).to(device)
print(f"Loaded | {sum(p.numel() for p in model_flan.parameters())/1e6:.0f}M params")

# ── Preprocessing ────────────────────────────────────────────
columns = [c for c in
    ["input","target","adl_type","element_type","quality","source"]
    if c in train_ds.column_names]

def preprocess_flan(examples):
    model_inputs = tokenizer_flan(
        examples["input"], max_length=128,
        truncation=True, padding=False)
    labels = tokenizer_flan(
        examples["target"], max_length=64,
        truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess_flan, batched=True, remove_columns=columns)
val_tok   = val_ds.map(preprocess_flan,   batched=True, remove_columns=columns)
print("Preprocessing OK")

# ── Training ─────────────────────────────────────────────────
args_flan = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_FLAN,
    num_train_epochs            = 50,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-4,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    optim                       = "adamw_torch",
    eval_strategy               = "epoch",
    logging_strategy            = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 5,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    predict_with_generate       = True,
    generation_max_length       = 64,
    fp16                        = False,
    bf16                        = False,
    report_to                   = "none",
    seed                        = 42,
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
)

trainer_flan = Seq2SeqTrainer(
    model            = model_flan,
    args             = args_flan,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    data_collator    = DataCollatorForSeq2Seq(
        tokenizer_flan, model=model_flan,
        padding=True, label_pad_token_id=-100),
    processing_class = tokenizer_flan,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.001)],
)

print(f"Starting FLAN-T5-base training — fp32 — {device}...")
trainer_flan.train()

# ── Save ─────────────────────────────────────────────────────
trainer_flan.save_model(OUTPUT_FLAN)
tokenizer_flan.save_pretrained(OUTPUT_FLAN)
print(f"Model saved      : {OUTPUT_FLAN}")
print(f"Stopped at epoch : {trainer_flan.state.epoch:.0f}/50")
print(f"Best eval_loss   : {trainer_flan.state.best_metric:.4f}")


In [ ]:
# ============================================================
# FLAN-T5-base — Resume from checkpoint (epoch ~25-29)
# A Colab disconnect interrupted the run above; this cell resumes
# training from the latest saved checkpoint.
# ============================================================
import os, gc, torch, glob, shutil
from transformers import (
    AutoTokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

OUTPUT_FLAN = f"{BASE_DIR}/finetuned_flan_t5_base"
device      = "cuda" if torch.cuda.is_available() else "cpu"

# ── 1. Find the latest valid checkpoint ──────────────────────
checkpoints = sorted(glob.glob(f"{OUTPUT_FLAN}/checkpoint-*"))
print(f"Available checkpoints: {checkpoints}")

# Remove older ones, keep only the latest
for ckpt in checkpoints[:-1]:
    shutil.rmtree(ckpt)
    print(f"Removed: {ckpt}")

last_checkpoint = checkpoints[-1]
print(f"✅ Resuming from: {last_checkpoint}")

gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── 2. Data ──────────────────────────────────────────────────
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# ── 3. Load tokenizer + model FROM the checkpoint ────────────
print(f"Loading model from checkpoint...")
tokenizer_flan = AutoTokenizer.from_pretrained(last_checkpoint)
model_flan = T5ForConditionalGeneration.from_pretrained(
    last_checkpoint, torch_dtype=torch.float32).to(device)
print(f"✅ {sum(p.numel() for p in model_flan.parameters())/1e6:.0f}M params loaded")

# ── 4. Preprocessing (identical to the initial run) ──────────
columns = [c for c in
    ["input","target","adl_type","element_type","quality","source"]
    if c in train_ds.column_names]

def preprocess_flan(examples):
    model_inputs = tokenizer_flan(
        examples["input"], max_length=128,
        truncation=True, padding=False)
    labels = tokenizer_flan(
        examples["target"], max_length=64,
        truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess_flan, batched=True, remove_columns=columns)
val_tok   = val_ds.map(preprocess_flan,   batched=True, remove_columns=columns)
print("✅ Preprocessing OK")

# ── 5. Arguments (save_total_limit reduced to 2) ─────────────
args_flan = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_FLAN,
    num_train_epochs            = 50,       # Trainer resumes at the correct epoch
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-4,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    optim                       = "adamw_torch",
    eval_strategy               = "epoch",
    logging_strategy            = "epoch",
    save_strategy                = "epoch",
    save_total_limit            = 2,        # ← reduced to avoid running out of disk
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    predict_with_generate       = True,
    generation_max_length       = 64,
    fp16                        = False,
    bf16                        = False,
    report_to                   = "none",
    seed                        = 42,
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
)

trainer_flan = Seq2SeqTrainer(
    model            = model_flan,
    args             = args_flan,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    data_collator    = DataCollatorForSeq2Seq(
        tokenizer_flan, model=model_flan,
        padding=True, label_pad_token_id=-100),
    processing_class = tokenizer_flan,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.001)],
)

# ── 6. Resume + guaranteed final save ────────────────────────
print(f"🚀 Resuming FLAN-T5-base training from {last_checkpoint}...")
trainer_flan.train(resume_from_checkpoint=last_checkpoint)

# Final save (outside the checkpoints directory)
trainer_flan.save_model(OUTPUT_FLAN)
tokenizer_flan.save_pretrained(OUTPUT_FLAN)
print(f"✅ Model permanently saved: {OUTPUT_FLAN}")
print(f"✅ Stopped at epoch : {trainer_flan.state.epoch:.0f}/50")
print(f"✅ Best loss        : {trainer_flan.state.best_metric:.4f}")


---
## 🟠 Model 5 — CodeT5+ 770M

**Larger code-specialized model.** Loaded in native fp16 with gradient checkpointing to fit the dataset/training run on a single T4 GPU.


In [ ]:
# ============================================================
# MODEL 5 — CodeT5+ 770M  ✅ STABLE VERSION
# ============================================================
import os, gc, torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Path parameters ──────────────────────────────────────────
DRIVE_CT5P  = "/content/drive/MyDrive/models/codet5p-770m"
OUTPUT_CT5P = f"{BASE_DIR}/finetuned_CodeT5plus_770m"
HF_MODEL_ID = "Salesforce/codet5p-770m"

os.makedirs(DRIVE_CT5P,  exist_ok=True)
os.makedirs(OUTPUT_CT5P, exist_ok=True)

# ── Free VRAM ────────────────────────────────────────────────
for _name in ["model_t5", "model_ct5", "model_flan", "model_uni", "model_ct5p", "model_m6"]:
    if _name in globals():
        try: globals()[_name].cpu()
        except: pass
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
print(f"🔋 Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Data ─────────────────────────────────────────────────────
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# ── Source detection (full Drive copy vs HuggingFace) ────────
# Checks both config.json AND the weights file to avoid a false positive
MODEL_FILE_ST  = os.path.join(DRIVE_CT5P, "model.safetensors")
MODEL_FILE_BIN = os.path.join(DRIVE_CT5P, "pytorch_model.bin")
CONFIG_FILE    = os.path.join(DRIVE_CT5P, "config.json")

model_on_drive_ok = (
    os.path.exists(CONFIG_FILE) and
    (os.path.exists(MODEL_FILE_ST) or os.path.exists(MODEL_FILE_BIN))
)
SOURCE = DRIVE_CT5P if model_on_drive_ok else HF_MODEL_ID

if model_on_drive_ok:
    print(f"✅ Full model found on Drive: {DRIVE_CT5P}")
else:
    print("⚠️  Model missing or incomplete — downloading from HuggingFace (~1.5 GB)...")

# ── Tokenizer ────────────────────────────────────────────────
print("⏳ Loading tokenizer...")
tokenizer_ct5p = AutoTokenizer.from_pretrained(
    SOURCE,
    use_fast         = False,                    # ✅ works around the extra_special_tokens bug
    local_files_only = (SOURCE != HF_MODEL_ID),
)
print("✅ Tokenizer loaded")

# ── Model ────────────────────────────────────────────────────
# torch_dtype=float16 -> model natively in fp16 on GPU (~1.5 GB VRAM)
# device_map="cuda"   -> GPU placement via accelerate
# tie_word_embeddings=False -> removes shared/lm_head/embed_tokens warnings
# fp16=False in TrainingArguments -> no redundant GradScaler
print("⏳ Loading CodeT5+ 770M (native fp16)...")
model_ct5p = AutoModelForSeq2SeqLM.from_pretrained(
    SOURCE,
    torch_dtype         = torch.float16,         # ✅ correct kwarg (dtype= was invalid)
    device_map          = "cuda",                # ✅ GPU placement via accelerate
    tie_word_embeddings = False,                 # ✅ removes warnings
)

# Gradient checkpointing + disable cache for training
model_ct5p.gradient_checkpointing_enable()
model_ct5p.config.use_cache = False

print(f"✅ CodeT5+ 770M loaded | "
      f"{sum(p.numel() for p in model_ct5p.parameters())/1e6:.0f}M params")
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Save to Drive on first download from HuggingFace
if SOURCE == HF_MODEL_ID:
    print(f"💾 Saving to Drive: {DRIVE_CT5P} ...")
    tokenizer_ct5p.save_pretrained(DRIVE_CT5P)
    model_ct5p.save_pretrained(DRIVE_CT5P)
    print("✅ Save complete — next sessions will load from Drive")

# ── Preprocessing ────────────────────────────────────────────
COLUMNS = [c for c in ["input", "target", "adl_type", "element_type", "quality", "source"]
           if c in train_ds.column_names]

def preprocess_ct5p(examples):
    model_inputs = tokenizer_ct5p(
        examples["input"],
        max_length=128, truncation=True
    )
    labels = tokenizer_ct5p(
        examples["target"],
        max_length=64, truncation=True
    )
    model_inputs["labels"] = [
        [(t if t != tokenizer_ct5p.pad_token_id else -100)
         for t in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_tok = train_ds.map(preprocess_ct5p, batched=True, remove_columns=COLUMNS)
val_tok   = val_ds.map(preprocess_ct5p,   batched=True, remove_columns=COLUMNS)
print("✅ Tokenization done")

# ── TrainingArguments ────────────────────────────────────────
args_ct5p = TrainingArguments(
    output_dir                  = OUTPUT_CT5P,
    num_train_epochs            = 20,

    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,

    learning_rate               = 1e-4,
    warmup_steps                = 100,
    weight_decay                = 0.01,

    # Adafactor + constant scheduler: stable combo for native fp16
    # (Adam with lr=1e-4 can diverge on unscaled fp16 gradients)
    optim             = "adafactor",
    lr_scheduler_type = "constant",

    eval_strategy                = "epoch",
    logging_strategy             = "epoch",
    save_strategy                = "epoch",
    save_total_limit            = 2,

    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    # ✅ fp16=False: the model is already fp16 at load time
    #    fp16=True would enable a redundant GradScaler -> ValueError
    fp16                        = False,
    bf16                        = False,

    report_to                   = "none",
    seed                        = 42,
    dataloader_pin_memory       = False,
)

# ── Trainer ──────────────────────────────────────────────────
trainer_ct5p = Trainer(
    model            = model_ct5p,
    args             = args_ct5p,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    data_collator    = DataCollatorForSeq2Seq(
        tokenizer_ct5p,
        model              = model_ct5p,
        padding            = True,
        label_pad_token_id = -100,
    ),
    processing_class = tokenizer_ct5p,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience  = 3,
        early_stopping_threshold = 0.001,
    )],
)

print("\n🚀 Starting CodeT5+ 770M fine-tuning...")
print("   native fp16 | grad_checkpointing | adafactor + constant LR")
print("   batch=2 | accum=4 | lr=1e-4 | max_input=128 | max_target=64\n")

trainer_ct5p.train()

print(f"\n✅ Stopped at epoch: {trainer_ct5p.state.epoch:.0f}/20")
print(f"✅ Best loss        : {trainer_ct5p.state.best_metric:.4f}")

# ── Save ─────────────────────────────────────────────────────
trainer_ct5p.save_model(OUTPUT_CT5P)
tokenizer_ct5p.save_pretrained(OUTPUT_CT5P)
print(f"✅ Saved: {OUTPUT_CT5P}")

# ── Re-enable cache before inference ─────────────────────────
# use_cache=False was required during training (gradient checkpointing)
# We re-enable it so beam search in evaluate_model() runs efficiently
model_ct5p.config.use_cache = True
if hasattr(model_ct5p, "generation_config"):
    model_ct5p.generation_config.use_cache = True
print("✅ Cache re-enabled for inference")

# ── Evaluation ───────────────────────────────────────────────
results_ct5p = evaluate_model(
    model_ct5p, tokenizer_ct5p,
    test_data,
    model_name = "CodeT5plus-770M",
    output_dir = OUTPUT_CT5P,
)


---
## ⚫ Model 6 — BART-large (~400M)

**Denoising-pretraining Encoder-Decoder**, included as an architectural contrast to the T5 family. Training and evaluation are combined in a single cell below.


In [ ]:
# ============================================================
# MODEL 6 — facebook/bart-large (~400M)
# Architecture: BART Encoder-Decoder (denoising pretraining)
# Purpose: architectural contrast against the T5 family on the ADL task
# Memory profile: ~1.6 GB fp32 -> use fp16 on the T4
# ============================================================
import os, gc, torch, json, numpy as np
from transformers import (
    BartTokenizerFast, BartForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from datasets import Dataset
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from bert_score import score as bert_score_fn
from nltk.translate.meteor_score import meteor_score
import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_BART  = "facebook/bart-large"
MODEL_NAME  = "BART-large"
OUTPUT_BART = f"{BASE_DIR}/finetuned_BART_large"
os.makedirs(OUTPUT_BART, exist_ok=True)

# ── Free VRAM ────────────────────────────────────────────────
for _name in ["model_t5","model_ct5","model_flan","model_uni","model_ct5p","model_bart"]:
    if _name in globals():
        try: globals()[_name].cpu()
        except: pass
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"🔋 Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print(f"⚙️  Device: {device} | Model: {MODEL_BART}")

# ── Data ─────────────────────────────────────────────────────
train_data, val_data, test_data = load_datasets()
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

# ── Tokenizer & Model ────────────────────────────────────────
print(f"\n⏳ Loading {MODEL_BART}...")
tokenizer_bart = BartTokenizerFast.from_pretrained(MODEL_BART)
model_bart     = BartForConditionalGeneration.from_pretrained(
    MODEL_BART,
    torch_dtype = torch.float16,  # native fp16: ~800 MB VRAM vs 1.6 GB fp32
).to(device)

# BART requires decoder_start_token_id = bos_token_id
model_bart.config.decoder_start_token_id = tokenizer_bart.bos_token_id
model_bart.config.forced_bos_token_id    = None  # avoids a generation conflict

# Gradient checkpointing to save VRAM
model_bart.gradient_checkpointing_enable()
model_bart.config.use_cache = False

n_params = sum(p.numel() for p in model_bart.parameters()) / 1e6
print(f"✅ {MODEL_NAME} loaded | {n_params:.0f}M params")
if torch.cuda.is_available():
    print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ── Preprocessing ────────────────────────────────────────────
# BART benefits from a task prefix (similar to FLAN) to steer
# the decoder toward generating the target ADL element
COLUMNS = [c for c in
    ["input","target","adl_type","element_type","quality","source"]
    if c in train_ds.column_names]

def preprocess_bart(examples):
    # Explicit prefix: helps BART contextualize the task
    inputs = ["understand ADL: " + str(x) for x in examples["input"]]
    model_inputs = tokenizer_bart(
        inputs,
        max_length  = 256,
        truncation  = True,
        padding     = False,
    )
    labels = tokenizer_bart(
        text_target = [str(t) for t in examples["target"]],
        max_length  = 128,
        truncation  = True,
        padding     = False,
    )
    # Mask padding in the loss
    model_inputs["labels"] = [
        [(t if t != tokenizer_bart.pad_token_id else -100)
         for t in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_tok = train_ds.map(preprocess_bart, batched=True, remove_columns=COLUMNS)
val_tok   = val_ds.map(preprocess_bart,   batched=True, remove_columns=COLUMNS)

# Sanity check on label integrity
ex = train_tok[0]
n_valid = sum(1 for t in ex["labels"] if t != -100)
print(f"\n🔍 Labels: {len(ex['labels'])} tokens, {n_valid} valid")
if n_valid == 0:
    raise ValueError("❌ Empty labels — check load_datasets()")
print("✅ Preprocessing OK")

# ── TrainingArguments ────────────────────────────────────────
# BART-large native fp16 -> fp16=False in the Trainer (no redundant GradScaler)
# adafactor + constant_lr: more stable than adamw with native fp16 on a small dataset
args_bart = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_BART,
    num_train_epochs            = 20,

    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,   # effective batch size = 8

    learning_rate               = 5e-5,   # BART converges better with a lower LR than T5
    warmup_steps                = 150,
    weight_decay                = 0.01,
    optim                       = "adafactor",
    lr_scheduler_type           = "constant",

    eval_strategy                = "epoch",
    logging_strategy             = "epoch",
    save_strategy                = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    predict_with_generate       = True,
    generation_max_length       = 128,
    generation_num_beams        = 4,

    fp16                        = False,  # model already fp16 at load time
    bf16                        = False,
    report_to                   = "none",
    seed                        = 42,
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
)

trainer_bart = Seq2SeqTrainer(
    model            = model_bart,
    args             = args_bart,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    data_collator    = DataCollatorForSeq2Seq(
        tokenizer_bart, model=model_bart,
        padding=True, label_pad_token_id=-100),
    processing_class = tokenizer_bart,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience  = 3,
        early_stopping_threshold = 0.001)],
)

print(f"\n🚀 Starting {MODEL_NAME} fine-tuning...")
print("   native fp16 | grad_checkpointing | adafactor + constant LR")
print("   batch=2 | accum=4 | lr=5e-5 | max_input=256 | max_target=128\n")

trainer_bart.train()

print(f"\n✅ Stopped at epoch: {trainer_bart.state.epoch:.0f}/20")
print(f"✅ Best loss       : {trainer_bart.state.best_metric:.4f}")

# ── Re-enable cache for inference ────────────────────────────
model_bart.config.use_cache = True
if hasattr(model_bart, "generation_config"):
    model_bart.generation_config.use_cache = True

# ── Save ─────────────────────────────────────────────────────
trainer_bart.save_model(OUTPUT_BART)
tokenizer_bart.save_pretrained(OUTPUT_BART)
print(f"✅ Model saved: {OUTPUT_BART}")

# ============================================================
# EVALUATION — BART-large
# ============================================================
print(f"\n🔄 Generating predictions for {len(test_data)} test examples...")
model_bart.eval()
predictions, references = [], []

for i, example in enumerate(test_data):
    inp = "understand ADL: " + str(example["input"])
    inputs = tokenizer_bart(
        inp, return_tensors="pt",
        max_length=256, truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model_bart.generate(
            **inputs,
            max_length           = 128,
            num_beams            = 4,
            early_stopping       = True,
            no_repeat_ngram_size = 3,
            length_penalty       = 1.0,
        )

    pred = tokenizer_bart.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred.strip())
    references.append(str(example["target"]).strip())

    if (i + 1) % 20 == 0:
        print(f"   {i+1}/{len(test_data)} generated...")

print(f"✅ {len(predictions)} predictions generated")

# ── Qualitative preview ──────────────────────────────────────
print("\n--- Preview: 3 examples ---")
for j in range(min(3, len(predictions))):
    print(f"\n  Reference : {references[j][:80]}")
    print(f"  Prediction: {predictions[j][:80]}")

# ── Exact Match ──────────────────────────────────────────────
em = sum(1 if p == r else 0
         for p, r in zip(predictions, references)) / len(predictions)

# ── BLEU ─────────────────────────────────────────────────────
smooth      = SmoothingFunction()
refs_bleu   = [[r.split()] for r in references]
preds_bleu  = [p.split()   for p in predictions]
bleu        = corpus_bleu(refs_bleu, preds_bleu, smoothing_function=smooth.method4)
smooth_bleu = corpus_bleu(refs_bleu, preds_bleu, smoothing_function=smooth.method1)

# ── ROUGE ────────────────────────────────────────────────────
scorer_r = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []
for p, r in zip(predictions, references):
    s = scorer_r.score(r, p)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)
rouge1 = float(np.mean(r1))
rouge2 = float(np.mean(r2))
rougeL = float(np.mean(rL))

# ── METEOR ───────────────────────────────────────────────────
meteor_scores = []
for p, r in zip(predictions, references):
    try:    meteor_scores.append(meteor_score([r.split()], p.split()))
    except: meteor_scores.append(0.0)
meteor = float(np.mean(meteor_scores))

# ── BERTScore ────────────────────────────────────────────────
print("\n🔄 Computing BERTScore...")
try:
    P, R, F1 = bert_score_fn(
        predictions, references,
        lang       = "en",
        model_type = "roberta-base",  # stable on a T4
        device     = device,
        batch_size = 16,
        verbose    = False,
    )
    bert_p  = float(P.mean().item())
    bert_r  = float(R.mean().item())
    bert_f1 = float(F1.mean().item())
except Exception as e:
    print(f"⚠️  BERTScore failed: {e}")
    bert_p = bert_r = bert_f1 = 0.0

# ── Final table ──────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"📊 RESULTS — {MODEL_NAME}")
print(f"   {len(test_data)} test examples")
print(f"{'='*60}")
for name, val in [
    ("Exact Match",    em),
    ("BLEU",           bleu),
    ("Smooth BLEU",    smooth_bleu),
    ("ROUGE-1",        rouge1),
    ("ROUGE-2",        rouge2),
    ("ROUGE-L",        rougeL),
    ("METEOR",         meteor),
    ("BERT-Precision", bert_p),
    ("BERT-Recall",    bert_r),
    ("BERT-F1",        bert_f1),
]:
    print(f"  {name:<20} : {val:.4f}")
print(f"{'='*60}")

# ── Save JSON ────────────────────────────────────────────────
results_bart = {
    "model"      : MODEL_NAME,
    "nb_test"    : len(test_data),
    "EM"         : em,
    "BLEU"       : bleu,
    "Smooth_BLEU": smooth_bleu,
    "ROUGE-1"    : rouge1,
    "ROUGE-2"    : rouge2,
    "ROUGE-L"    : rougeL,
    "METEOR"     : meteor,
    "BERT-P"     : bert_p,
    "BERT-R"     : bert_r,
    "BERT-F1"    : bert_f1,
}
path_json = f"{OUTPUT_BART}/results_{MODEL_NAME.replace(' ','_')}.json"
with open(path_json, "w", encoding="utf-8") as f:
    json.dump(results_bart, f, indent=4, ensure_ascii=False)
print(f"\n💾 Results saved: {path_json}")


---
## 📊 Step 2 — Unified evaluation across all 6 backbones

Reloads each fine-tuned model from disk and recomputes the full metric suite (EM, BLEU, ROUGE-1/2/L, METEOR, BERTScore) on the same held-out test set, producing the comparison table used in the paper.


In [ ]:
# ============================================================
# UNIFIED EVALUATION — all 6 comprehension backbones
# Metrics: EM, BLEU, ROUGE-1/2/L, METEOR, BERT-F1
# ============================================================
import os, gc, time, json, torch, numpy as np
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BartForConditionalGeneration, BartTokenizerFast,
    EncoderDecoderModel, RobertaTokenizer,
)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from bert_score import score as bert_score_fn
from nltk.translate.meteor_score import meteor_score
import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

OUT_EVAL = f"{BASE_DIR}/evaluation_comprehension"
os.makedirs(OUT_EVAL, exist_ok=True)

# ── Test data ────────────────────────────────────────────────
_, _, test_data = load_datasets()
print(f"✅ Test set: {len(test_data)} examples\n")

# ── Generic generation routine ──────────────────────────────
def generate_predictions(model, tokenizer, test_data,
                          max_input=256, max_target=128,
                          decoder_start_token_id=None):
    model.eval()
    preds, refs = [], []
    gen_kwargs = dict(max_length=max_target, num_beams=4,
                      early_stopping=True)
    if decoder_start_token_id is not None:
        gen_kwargs["decoder_start_token_id"] = decoder_start_token_id
    for i, ex in enumerate(test_data):
        inputs = tokenizer(
            str(ex["input"]), return_tensors="pt",
            max_length=max_input, truncation=True
        ).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        preds.append(pred)
        refs.append(str(ex["target"]).strip())
        if (i + 1) % 50 == 0:
            print(f"     {i+1}/{len(test_data)} generated...")
    return preds, refs

# ── Full metric computation ─────────────────────────────────
def compute_metrics(preds, refs, name):
    # Exact Match
    em = sum(p == r for p, r in zip(preds, refs)) / len(preds)

    # BLEU
    smooth  = SmoothingFunction()
    refs_b  = [[r.split()] for r in refs]
    preds_b = [p.split() for p in preds]
    bleu    = corpus_bleu(refs_b, preds_b,
                          smoothing_function=smooth.method4)

    # ROUGE-1 / ROUGE-2 / ROUGE-L
    scorer = rs_module.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)

    # METEOR
    met = float(np.mean([
        meteor_score([r.split()], p.split())
        for p, r in zip(preds, refs)
    ]))

    # BERTScore
    print(f"     🔄 BERTScore...")
    P, R, F1 = bert_score_fn(
        preds, refs, lang="en",
        model_type="roberta-base", verbose=False)

    return {
        "model"   : name,
        "nb_test" : len(preds),
        "EM"      : round(float(em),            4),
        "BLEU"    : round(float(bleu),           4),
        "ROUGE-1" : round(float(np.mean(r1)),    4),
        "ROUGE-2" : round(float(np.mean(r2)),    4),
        "ROUGE-L" : round(float(np.mean(rL)),    4),
        "METEOR"  : round(float(met),            4),
        "BERT-F1" : round(float(F1.mean()),      4),
        "BERT-P"  : round(float(P.mean()),       4),
        "BERT-R"  : round(float(R.mean()),       4),
    }

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Registry of the 6 models ────────────────────────────────
MODELS = [
    {"id": "M1", "name": "T5-base",
     "path": f"{BASE_DIR}/finetuned_T5base",
     "type": "t5"},
    {"id": "M2", "name": "CodeT5-base",
     "path": f"{BASE_DIR}/finetuned_CodeT5base",
     "type": "t5"},
    {"id": "M3", "name": "UniXcoder",
     "path": f"{BASE_DIR}/finetuned_UniXcoder",
     "type": "unixcoder"},
    {"id": "M4", "name": "Flan-T5-base",
     "path": f"{BASE_DIR}/finetuned_flan_t5_base",
     "type": "t5"},
    {"id": "M5", "name": "CodeT5plus-770M",
     "path": f"{BASE_DIR}/finetuned_CodeT5plus_770m",
     "type": "seq2seq"},
    {"id": "M6", "name": "BART-large",
     "path": f"{BASE_DIR}/finetuned_BART_large",
     "type": "bart"},
]

# ── Main loop ────────────────────────────────────────────────
results = {}   # mid -> metrics dict

for cfg in MODELS:
    mid, name, path, mtype = (cfg["id"], cfg["name"],
                               cfg["path"], cfg["type"])
    print(f"\n{'='*60}")
    print(f"  [{mid}] {name}")
    print(f"{'='*60}")

    if not os.path.exists(path):
        print(f"  ⚠️  Not found: {path} — skipped")
        continue

    t0 = time.time()
    decoder_start = None
    try:
        if mtype in ("t5", "seq2seq"):
            tok   = AutoTokenizer.from_pretrained(path)
            model = AutoModelForSeq2SeqLM.from_pretrained(
                path, torch_dtype=torch.float32).to(device)
        elif mtype == "bart":
            tok   = BartTokenizerFast.from_pretrained(path)
            model = BartForConditionalGeneration.from_pretrained(
                path, torch_dtype=torch.float32).to(device)
        elif mtype == "unixcoder":
            tok   = RobertaTokenizer.from_pretrained(path)
            model = EncoderDecoderModel.from_pretrained(
                path).to(device)
            decoder_start = tok.cls_token_id

        n_params = sum(p.numel() for p in model.parameters())/1e6
        print(f"  ✅ Loaded | {n_params:.0f}M parameters")
        print(f"  🔄 Generating on {len(test_data)} examples...")

        preds, refs = generate_predictions(
            model, tok, test_data,
            decoder_start_token_id=decoder_start)

        print(f"  📊 Computing metrics...")
        metrics = compute_metrics(preds, refs, name)
        metrics["id"]      = mid
        metrics["latency_s"] = round(time.time() - t0, 1)
        results[mid]       = metrics

        print(f"\n  📈 [{mid}] {name}")
        for k in ["EM","BLEU","ROUGE-1","ROUGE-2",
                  "ROUGE-L","METEOR","BERT-F1"]:
            print(f"     {k:<12}: {metrics[k]:.4f}")

        fpath = f"{OUT_EVAL}/results_{mid}_{name}.json"
        with open(fpath, "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"  💾 {fpath}")

    except Exception as e:
        print(f"  ❌ Error: {e}")
        import traceback; traceback.print_exc()
    finally:
        try: del model, tok
        except: pass
        free_gpu()
        print(f"  🧹 Memory freed")

# ── SUMMARY TABLE ────────────────────────────────────────────
DISPLAY_METRICS = ["EM","BLEU","ROUGE-1","ROUGE-2",
                   "ROUGE-L","METEOR","BERT-F1"]

print(f"\n\n{'='*95}")
print(f"  SUMMARY TABLE — {len(results)}/{len(MODELS)} models")
print(f"{'='*95}")

# Header
hdr = f"  {'Model':<20}"
for m in DISPLAY_METRICS:
    hdr += f"  {m:>8}"
print(hdr)
print("  " + "-"*20 + ("  " + "-"*8) * len(DISPLAY_METRICS))

# Rows
for mid in ["M1","M2","M3","M4","M5","M6"]:
    if mid not in results:
        continue
    r = results[mid]
    line = f"  {r['model']:<20}"
    for m in DISPLAY_METRICS:
        line += f"  {r[m]:>8.4f}"
    print(line)

print("  " + "-"*20 + ("  " + "-"*8) * len(DISPLAY_METRICS))

# Best per column
best_line = f"  {'Best ↑':<20}"
for m in DISPLAY_METRICS:
    best_val = max(results[mid][m] for mid in results)
    best_line += f"  {best_val:>8.4f}"
print(best_line)
print(f"{'='*95}")

# Save the global recap
recap_path = f"{OUT_EVAL}/recap_global.json"
with open(recap_path, "w") as f:
    json.dump(list(results.values()), f, indent=2)
print(f"\n💾 Recap saved: {recap_path}")


---
##  Step 3 — Weight sensitivity analysis & backbone selection

Tests whether the Top-1/Top-2 backbone ranking is robust to the choice of metric-weighting scheme, then selects the two backbones carried forward to the evolution fine-tuning stage (notebook 04).


In [ ]:
# ============================================================
# WEIGHT SENSITIVITY ANALYSIS + BACKBONE SELECTION
# Reloads results from the JSON files saved on Drive
# ============================================================
import numpy as np, json, os

OUT_EVAL = f"{BASE_DIR}/evaluation_comprehension"

# ── Reload from JSON ─────────────────────────────────────────
FILES = {
    "M1": "results_M1_T5-base.json",
    "M2": "results_M2_CodeT5-base.json",
    "M3": "results_M3_UniXcoder.json",
    "M4": "results_M4_Flan-T5-base.json",
    "M5": "results_M5_CodeT5plus-770M.json",
    "M6": "results_M6_BART-large.json",
}

results = {}
for mid, fname in FILES.items():
    fpath = f"{OUT_EVAL}/{fname}"
    if os.path.exists(fpath):
        with open(fpath) as f:
            results[mid] = json.load(f)
        print(f"✅ {mid} reloaded: {fname}")
    else:
        print(f"❌ Not found: {fpath}")

if not results:
    raise RuntimeError("No results found on Drive.")
print(f"\n✅ {len(results)}/6 models reloaded\n")

# ── Metric extraction ────────────────────────────────────────
def extract(res):
    return {
        "em"      : res.get("EM",      0.0),
        "bleu"    : res.get("BLEU",    0.0),
        "rouge_l" : res.get("ROUGE-L", 0.0),
        "meteor"  : res.get("METEOR",  0.0),
        "bert_f1" : res.get("BERT-F1", 0.0),
    }

metrics = {res["model"]: extract(res)
           for res in results.values()}

# ── 6 weight configurations ──────────────────────────────────
CONFIGS = {
    "C1-Nominal     (EM·0.25 BL·0.20 RL·0.20 ME·0.15 BF·0.20)": {
        "em":0.25,"bleu":0.20,"rouge_l":0.20,
        "meteor":0.15,"bert_f1":0.20},
    "C2-Uniform     (0.20 each)": {
        "em":0.20,"bleu":0.20,"rouge_l":0.20,
        "meteor":0.20,"bert_f1":0.20},
    "C3-BERT dom.   (BF·0.50)": {
        "em":0.15,"bleu":0.15,"rouge_l":0.15,
        "meteor":0.05,"bert_f1":0.50},
    "C4-EM dom.     (EM·0.50)": {
        "em":0.50,"bleu":0.15,"rouge_l":0.15,
        "meteor":0.10,"bert_f1":0.10},
    "C5-NLP only    (no EM)": {
        "em":0.00,"bleu":0.30,"rouge_l":0.30,
        "meteor":0.20,"bert_f1":0.20},
    "C6-Structural  (EM·0.40 BL·0.30)": {
        "em":0.40,"bleu":0.30,"rouge_l":0.15,
        "meteor":0.05,"bert_f1":0.10},
}

for cname, weights in CONFIGS.items():
    s = sum(weights.values())
    assert abs(s-1.0) < 1e-9, f"Invalid weights {cname}: {s:.4f}"

# ── Ranking computation ──────────────────────────────────────
def compute_score(metr, weights):
    return round(sum(weights[k]*metr[k] for k in weights), 4)

ref_config  = list(CONFIGS.keys())[0]
sensitivity_results = {}

for cname, weights in CONFIGS.items():
    scores = {name: compute_score(m, weights) for name, m in metrics.items()}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    sensitivity_results[cname] = {
        "scores"  : scores,
        "ranking" : [n for n,_ in ranked],
        "top1"    : ranked[0][0],
        "top2"    : ranked[1][0] if len(ranked)>1 else None,
    }

top1_ref = sensitivity_results[ref_config]["top1"]
top2_ref = sensitivity_results[ref_config]["top2"]
models   = list(metrics.keys())

# ── SENSITIVITY TABLE ────────────────────────────────────────
print("="*100)
print("  📊 WEIGHT SENSITIVITY ANALYSIS")
print("="*100)
hdr = f"  {'Configuration':<52}"
for name in models:
    hdr += f"  {name[:13]:>13}"
hdr += f"  {'Top-1':<18} {'Stable'}"
print(hdr)
print("  "+"-"*52+("  "+"-"*13)*len(models)+"  "+"-"*18+"  "+"-"*6)

for cname, res in sensitivity_results.items():
    stable = "✅" if res["top1"] == top1_ref else "⚠️ "
    line  = f"  {cname:<52}"
    for name in models:
        sc  = res["scores"][name]
        tag = "*" if name == res["top1"] else " "
        line += f"  {tag+f'{sc:.4f}':>13}"
    line += f"  {res['top1']:<18} {stable}"
    print(line)

print("  "+"-"*52+("  "+"-"*13)*len(models)+"  "+"-"*18+"  "+"-"*6)
print("  * = best score | ✅ = stable Top-1 | ⚠️  = different")

# ── STABILITY REPORT ──────────────────────────────────────────
num_configs = len(CONFIGS)
num_stable  = sum(1 for r in sensitivity_results.values()
                  if r["top1"] == top1_ref)
stability_rate = num_stable / num_configs * 100

margins = []
for res in sensitivity_results.values():
    cl = res["ranking"]
    if len(cl) >= 2:
        margins.append(res["scores"][cl[0]] - res["scores"][cl[1]])

top1_scores_all = [r["scores"][top1_ref] for r in sensitivity_results.values()]
top1_spread = max(top1_scores_all) - min(top1_scores_all)

print(f"\n{'='*65}")
print(f"  📋 STABILITY REPORT")
print(f"{'='*65}")
print(f"  Top-1 backbone (nominal)     : {top1_ref}")
print(f"  Top-2 backbone (nominal)     : {top2_ref}")
print(f"  Configurations tested        : {num_configs}")
print(f"  Stable Top-1                 : {num_stable}/{num_configs} ({stability_rate:.0f}%)")
print(f"  Top-1 score spread (all cfg) : {top1_spread:.4f}")
print(f"  Mean margin Top1 vs Top2     : {float(np.mean(margins)):.4f}")
print(f"  Min margin Top1 vs Top2      : {float(np.min(margins)):.4f}")

# ── SELECT TOP 2 BACKBONES ───────────────────────────────────
ref_scores  = sensitivity_results[ref_config]["scores"]
top2_select = sorted(ref_scores.items(),
                     key=lambda x: x[1], reverse=True)[:2]

print(f"\n{'='*65}")
print(f"  🏆 BACKBONES SELECTED FOR THE COMPREHENSION TASK")
print(f"{'='*65}")
for rank, (name, score) in enumerate(top2_select, 1):
    m = metrics[name]
    print(f"\n  #{rank} — {name}")
    print(f"       Composite score : {score:.4f}")
    print(f"       EM={m['em']:.4f} | BLEU={m['bleu']:.4f} | "
          f"ROUGE-L={m['rouge_l']:.4f} | "
          f"METEOR={m['meteor']:.4f} | BERT-F1={m['bert_f1']:.4f}")

# ── SENTENCE FOR THE PAPER ───────────────────────────────────
b1, b2 = top2_select[0][0], top2_select[1][0]
s1, s2 = top2_select[0][1], top2_select[1][1]
min_margin = float(np.min(margins))

print(f"\n{'='*65}")
print(f"  📝 SENTENCE FOR THE PAPER")
print(f"{'='*65}")
print(f"""
  "To select the backbone(s) for the architectural evolution
   task, we rank all {len(models)} fine-tuned comprehension models
   using a composite score:

       S = 0.25·EM + 0.20·BLEU + 0.20·ROUGE-L
         + 0.15·METEOR + 0.20·BERT-F1

   A sensitivity analysis across {num_configs} weight configurations
   (uniform, BERT-dominant, EM-dominant, NLP-only, structure-
   first) confirms that {b1} ranks first in {num_stable} of
   {num_configs} configurations (stability: {stability_rate:.0f}%, minimum
   margin over runner-up: {min_margin:.4f}), validating that the
   selection is robust to the weighting scheme.

   Accordingly, {b1} (S={s1:.4f}) and {b2} (S={s2:.4f})
   are retained as backbone candidates for the evolution
   fine-tuning stage."
""")

# ── SAVE ─────────────────────────────────────────────────────
report = {
    "selected_backbones" : [b1, b2],
    "nominal_scores"      : ref_scores,
    "stability_pct"       : stability_rate,
    "min_margin_top1_top2": min_margin,
    "mean_margin_top1_top2": float(np.mean(margins)),
    "top1_score_spread"   : top1_spread,
    "configs"             : {
        cname: {
            "weights" : CONFIGS[cname],
            "top1"    : res["top1"],
            "top2"    : res["top2"],
            "scores"  : res["scores"],
            "ranking" : res["ranking"],
        }
        for cname, res in sensitivity_results.items()
    }
}
path_out = f"{OUT_EVAL}/sensitivity_backbone_selection.json"
with open(path_out, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"  💾 Report: {path_out}")
